# GW analysis of monolayer and bilayer hBN

Yambo (Lumen fork, `bug-fixes` build) G0W0 calculations for the first benchmark of the project
(`docs/04_benchmark_systems.md`): explicit bilayer hBN vs monolayer + second layer as an effective substrate.
The GW parameters are converged on the monolayer (`ML_bot`) and used unchanged for all the systems.

Choices (user, 2026-11-11):
- k grids 18x18x1, 30x30x1, 42x42x1 (they contain K and M), nscf with 250 bands: upper bound of all the band
  parameters of the GW runs (`DFT_Analysis.ipynb`, section "NSCF on regular grids").
- Coulomb interaction: slab cutoff `CUTGeo='slab z'` and RIM on v in every run, RandQpts = 5e6, RandGvec = 100
  (values of the MoS2 study, not re-converged).
- HF runs with RIM only; all the GW (PPA) runs with RIM_W, RandGvecW = 10.

How to run: execute the cells in order; every dataset uses `skip=True` (existing results are not recomputed) and at
most 2 slurm jobs at a time (project rule: at most 2 nodes).

In [1]:
# useful to autoreload the module without restarting the kernel
%load_ext autoreload
%autoreload 2

In [2]:
from mppi import InputFiles as I, Calculators as C, Datasets as D, Parsers as P, Utilities as U
from mppi.Calculators import Tools
from mppi.Datasets import PostProcessing as PP
from mppi.Utilities import Constants as Const
import matplotlib.pyplot as plt
import numpy as np
import os

In [3]:
# Calculator factory for ismhpc (32 cores per node, 32 MPI x 1 OMP for yambo)
module = '/home/dalessandro/module_script/yambo_module'

def yambo_code(nodes=1, partition='all12h', time='11:59:00', executable='yambo'):
    rr = C.RunRules(scheduler='slurm', partition=partition, memory='125000', time=time, nodes=nodes,
                    ntasks_per_node=32, cpus_per_task=1, omp_num_threads=1, pre_processing=module)
    return C.YamboCalculator(rr, executable=executable, activate_BeeOND=True)

## SAVE databases
One yambo folder per nscf: `MB_analysis/<nscf prefix>/SAVE`, built with `Tools.init_yambo_dir` (p2y + yambo setup
`r_setup`). p2y and the setup are serial and light: they run directly on the login node with 1 MPI and 1 OMP (user,
2026-11-11). Folders whose SAVE and `r_setup` already exist are skipped. The Lumen p2y ends with MPI_ABORT after
writing the databases: the SAVE is complete anyway (checked below). (Launching the OpenMPI p2y with `srun` hangs at
start-up on ismhpc.)

In [4]:
dft_dir = 'DFT_analysis'
mb_dir = 'MB_analysis'
saves = [('ML_bot', 18, 250), ('ML_bot', 30, 250), ('ML_bot', 42, 250),   # (system, k, bands) of the nscf runs
         ('AA1', 18, 400), ('AB', 18, 400)]

def nscf_name(system, k, nbnds):
    return 'kpoints_%s-nbnds_%s-%s' % (k, nbnds, system)

def yambo_dir(system, k, nbnds):
    return os.path.join(mb_dir, nscf_name(system, k, nbnds))

def build_save(system, k, nbnds):
    Tools.init_yambo_dir(yambo_dir=yambo_dir(system, k, nbnds),
                         input_dir=os.path.join(dft_dir, nscf_name(system, k, nbnds) + '.save'),
                         p2y_command='OMP_NUM_THREADS=1 mpirun -np 1 p2y',
                         yambo_command='OMP_NUM_THREADS=1 mpirun -np 1 yambo')

for s, k, nb in saves:
    build_save(s, k, nb)

SAVE folder MB_analysis/kpoints_18-nbnds_250-ML_bot/SAVE already present. No operations performed.
SAVE folder MB_analysis/kpoints_30-nbnds_250-ML_bot/SAVE already present. No operations performed.
SAVE folder MB_analysis/kpoints_42-nbnds_250-ML_bot/SAVE already present. No operations performed.
Create the folder path MB_analysis/kpoints_18-nbnds_400-AA1
Executing command: cd MB_analysis/kpoints_18-nbnds_400-AA1; OMP_NUM_THREADS=1 mpirun -np 1 p2y -I ../../DFT_analysis/kpoints_18-nbnds_400-AA1.save


Build the r_setup in the yambo_dir path MB_analysis/kpoints_18-nbnds_400-AA1


Create the folder path MB_analysis/kpoints_18-nbnds_400-AB
Executing command: cd MB_analysis/kpoints_18-nbnds_400-AB; OMP_NUM_THREADS=1 mpirun -np 1 p2y -I ../../DFT_analysis/kpoints_18-nbnds_400-AB.save


Build the r_setup in the yambo_dir path MB_analysis/kpoints_18-nbnds_400-AB


In [5]:
def check_save(system, k, nbnds):
    folder = yambo_dir(system, k, nbnds)
    save = os.path.join(folder, 'SAVE')
    nwf = len([f for f in os.listdir(save) if f.startswith('ns.wf')])
    db = P.YamboDftParser(os.path.join(save, 'ns.db1'))
    with open(os.path.join(folder, 'r_setup')) as f: setup = f.read()
    print('%-30s %4d k points (IBZ) | %d bands | %d ns.wf files | r_setup %s'
          % (folder, len(db.get_kpoints()), db.nbands, nwf, 'ok' if 'Game Over' in setup else 'INCOMPLETE'))

for s, k, nb in saves:
    check_save(s, k, nb)

Parse file : MB_analysis/kpoints_18-nbnds_250-ML_bot/SAVE/ns.db1
MB_analysis/kpoints_18-nbnds_250-ML_bot   37 k points (IBZ) | 250 bands | 38 ns.wf files | r_setup ok
Parse file : MB_analysis/kpoints_30-nbnds_250-ML_bot/SAVE/ns.db1
MB_analysis/kpoints_30-nbnds_250-ML_bot   91 k points (IBZ) | 250 bands | 92 ns.wf files | r_setup ok
Parse file : MB_analysis/kpoints_42-nbnds_250-ML_bot/SAVE/ns.db1
MB_analysis/kpoints_42-nbnds_250-ML_bot  169 k points (IBZ) | 250 bands | 170 ns.wf files | r_setup ok
Parse file : MB_analysis/kpoints_18-nbnds_400-AA1/SAVE/ns.db1
MB_analysis/kpoints_18-nbnds_400-AA1   37 k points (IBZ) | 400 bands | 38 ns.wf files | r_setup ok
Parse file : MB_analysis/kpoints_18-nbnds_400-AB/SAVE/ns.db1
MB_analysis/kpoints_18-nbnds_400-AB   37 k points (IBZ) | 400 bands | 38 ns.wf files | r_setup ok
